# Config – Metadata Driven JSON Loader


Two tables drive this pipeline:
1. **source_control** – the sources to process (source, file_path,
   file_type, target catalog/schema/table, is_active)
2. **load_logs** – every run and every file attempt gets a row here. On a
   re-run, this table is checked to work out which files already
   succeeded and which still need loading

## Required Libraries

In [0]:
from pyspark.sql.functions import col, parse_json, lit, current_timestamp, desc, row_number, json_object_keys, array
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, TimestampType, ArrayType
import uuid
from datetime import datetime

## Meta table locations

In [0]:
meta_catalog = "iics"
meta_schema  = "casemix"

source_control_table = "json_control_test"   # source config
log_table             = "json_logs_test"        # audit log
file_inventory_table  = "json_file_inventory_test"

#executing user
executed_by = spark.sql("SELECT current_user()").first()[0]


## Run identifiers

Every log row from one execution shares the same `run_id`.

In [0]:
notebook_name = "02_main"
run_id = str(uuid.uuid4())